# 4. A Multi-Harness MemAgent Review Team

This notebook creates a real MemoRizz multi-agent topology in which two `MemAgent` delegates use different external harnesses:

- a **Codex-backed correctness analyst** examines behavior, edge cases, and tests; and
- a **Claude Code-backed adversarial reviewer** independently examines API safety, failure modes, and maintainability.

A third `MemAgent` is the coordinator. It owns a deterministic delegation plan, tenant scope, shared workflow memory, trace identity, and the consolidated report. Planning stays deterministic; when execution is explicitly enabled, the coordinator uses a configured native `LLMProvider` for the final reconciliation. MemoRizz MetaHarness owns each vendor run's policy, bounded memory context, normalized events, verification, and durable evidence.

> **Default behavior:** this notebook builds and validates the complete multi-agent topology but does not call either vendor. Set `MEMORIZZ_RUN_MULTI_HARNESS_DEMO=1` before launching Jupyter to execute both read-only reviews.

## Why use multiple harnesses?

Using two harnesses is justified when role diversity is expected to reduce correlated mistakes. It is not a default recipe for every task. This example uses **blind, independent review**: neither reviewer sees the other's answer while reasoning, which reduces anchoring and lets their reports run in parallel. The coordinator reconciles them afterward.

```mermaid
flowchart TB
    Request[High-impact review request] --> Coordinator[Coordinator MemAgent]
    Coordinator --> Plan[Deterministic scoped plan]
    Plan --> CodexAgent[Correctness MemAgent]
    Plan --> ClaudeAgent[Adversarial MemAgent]
    CodexAgent --> Codex[Codex harness]
    ClaudeAgent --> Claude[Claude Code harness]
    Memory[(Shared tenant-scoped memory)] --> Codex
    Memory --> Claude
    Codex --> CodexEvidence[Verified run evidence]
    Claude --> ClaudeEvidence[Verified run evidence]
    CodexEvidence & ClaudeEvidence --> Reconcile[Coordinator reconciliation]
    Reconcile --> Decision[Agreements, disagreements, next action]
```

| Benefit | Why it can help | Guardrail in this notebook |
|---|---|---|
| Different failure modes | Models and harness tool loops may notice different defects | Give each a distinct review role rather than duplicate prompts |
| Independent cross-check | Agreement increases confidence; disagreement identifies uncertainty | Preserve both original reports and provenance |
| Provider resilience | One provider failure need not erase the other's evidence | Report partial failures explicitly |
| Parallel latency | Independent read-only tasks can execute concurrently | Do not parallelize writes to one workspace |
| Better learning evidence | Repeated verified agreement is stronger than one unverified trajectory | Promote only after outcome and diversity gates |

The cost is roughly two model runs plus reconciliation. Prefer one harness for routine, low-risk work; use a panel for high-impact changes, ambiguous failures, security reviews, or evaluation.

## Three layers of orchestration

```mermaid
sequenceDiagram
    participant U as Host application
    participant R as Coordinator MemAgent
    participant O as MultiAgentOrchestrator
    participant C as Codex-backed MemAgent
    participant A as Claude-backed MemAgent
    participant M as MetaHarness
    U->>R: run(query, memory_id, user_id, thread_id)
    R->>O: deterministic SubTask plan
    par Independent correctness review
        O->>C: delegated task + scope + trace
        C->>M: runtime task, harness=codex
    and Independent adversarial review
        O->>A: delegated task + scope + trace
        A->>M: runtime task, harness=claude-code
    end
    M-->>O: normalized verified evidence
    O-->>R: task report + partial failure states
    R-->>U: consolidated response and workflow report
```

The multi-agent orchestrator coordinates participants and dependencies. MetaHarness governs external processes. Keeping these responsibilities separate means adding a third vendor does not require redesigning memory scope, approval semantics, or observability.

## 1. Create a review fixture and scoped memory

The sample module is intentionally small so a real opt-in run is inexpensive. It has an explicit contract and a host verification script. The memory record defines review expectations and is scoped to one `memory_id`, `user_id`, and `thread_id`; both delegates receive the same source provenance.

No credentials are loaded or displayed by this notebook. Configure the vendor CLIs outside Jupyter, then confirm readiness with `memorizz harness doctor`.

In [ ]:
import os
import shutil
import sys
import tempfile
from pathlib import Path
from pprint import pprint

from memorizz import MemAgentBuilder
from memorizz.approval import SQLiteApprovalStore
from memorizz.enums.memory_type import MemoryType
from memorizz.memory_provider.filesystem.provider import FileSystemConfig, FileSystemProvider
from memorizz.metaharness import MetaHarness, SQLiteHarnessRunStore

RUN_MULTI_HARNESS_DEMO = os.getenv("MEMORIZZ_RUN_MULTI_HARNESS_DEMO", "").strip().lower() in {"1", "true", "yes", "on"}

DEMO_ROOT = Path(tempfile.mkdtemp(prefix="memorizz-multi-harness-team-"))
WORKSPACE = DEMO_ROOT / "workspace"
WORKSPACE.mkdir()
(WORKSPACE / "metrics.py").write_text(
    'def ratio(total: float, count: int) -> float:\n'
    '    """Return total/count; count must be a positive integer."""\n'
    '    if isinstance(count, bool) or not isinstance(count, int) or count <= 0:\n'
    '        raise ValueError("count must be a positive integer")\n'
    '    return total / count\n',
    encoding="utf-8",
)
(WORKSPACE / "verify.py").write_text(
    'from metrics import ratio\n'
    'assert ratio(10.0, 2) == 5.0\n'
    'for invalid in (0, -1, True, 1.5):\n'
    '    try:\n'
    '        ratio(10.0, invalid)\n'
    '    except ValueError:\n'
    '        pass\n'
    '    else:\n'
    '        raise AssertionError(f"accepted invalid count: {invalid!r}")\n',
    encoding="utf-8",
)

provider = FileSystemProvider(
    FileSystemConfig(
        root_path=DEMO_ROOT / "memory",
        embedding_provider=None,
        lazy_vector_indexes=True,
        use_faiss=False,
    )
)
MEMORY_ID = "metrics-review"
USER_ID = "tutorial-reviewer"
THREAD_ID = "change-128"
CODEX_REVIEW_TASK = (
    "Independently review metrics.py and verify.py for correctness, type edge "
    "cases, invariants, and missing tests. Do not modify files. Cite MemoRizz "
    "memory source IDs when using retrieved requirements. Return findings with "
    "severity and concrete evidence."
)
CLAUDE_REVIEW_TASK = (
    "Independently and adversarially review metrics.py and verify.py for ambiguous "
    "contracts, misuse cases, failure semantics, maintainability, and untested risk. "
    "Do not modify files. Cite MemoRizz memory source IDs when using retrieved "
    "requirements. Return findings with severity and concrete evidence."
)
memory_source_id = provider.store(
    {
        "title": "Metrics API review requirements",
        "content": (
            f"Applicable request: {CODEX_REVIEW_TASK}\n"
            f"Applicable request: {CLAUDE_REVIEW_TASK}\n"
            "Review metrics.py for numerical correctness, Python bool/int edge cases, "
            "clear failure behavior, public API stability, and test coverage. The review "
            "must not modify files and should distinguish findings from suggestions."
        ),
        "user_id": USER_ID,
        "thread_id": THREAD_ID,
    },
    MemoryType.KNOWLEDGE_BASE,
    memory_id=MEMORY_ID,
)
print({"workspace": str(WORKSPACE), "memory_source_id": memory_source_id})

## 2. Create one shared MetaHarness control plane

Both delegates share one `MetaHarness` instance and therefore one run ledger, approval store, memory context builder, router, and workspace policy boundary. They do **not** share vendor sessions or hidden model state. Each run remains independently attributable by harness, agent ID, user, thread, source IDs, and trace.

In [ ]:
service = MetaHarness.from_env(
    memory_provider=provider,
    run_store=SQLiteHarnessRunStore(DEMO_ROOT / "runs.sqlite3"),
    approval_store=SQLiteApprovalStore(DEMO_ROOT / "approvals.sqlite3"),
    allowed_workspace_roots=[str(WORKSPACE)],
)

required_harnesses = ["codex", "claude-code"]
probes = {name: service.probe(name) for name in required_harnesses}
for name, probe in probes.items():
    print(f"{name:<12} ready={probe['ready']} version={probe.get('version')} error={probe.get('error')}")

## 3. Wrap each harness in a role-specific MemAgent

The participants differ in named identity, explicit delegated task, and execution harness. Runtime mode builds the external prompt from the task envelope, scoped memory, and host contract, so each role is stated in its deterministic `SubTask` rather than relying on a native MemAgent system prompt. They share the same conservative boundary: read-only workspace, no network, no MCP injection, bounded steps/tokens, and a host verification command. The twelve-action limit is deliberately small for two fixture files while allowing normal inspect/probe/verify lifecycles; MemoRizz counts a vendor action once even when its stream emits both `in_progress` and `completed` records. The 8,000-token ceiling covers provider-reported multi-turn reasoning and tool usage, not a target length for the final prose. `python -B` prevents verification from writing bytecode into the read-only fixture.

A role should change what evidence an agent seeks. If both prompts simply say “review this code,” two harnesses often buy cost without meaningful diversity.

In [ ]:
VERIFICATION = f'"{sys.executable}" -B verify.py'
COMMON_CONFIG = {
    "workspace": str(WORKSPACE),
    "permissions": {
        "workspace_mode": "read_only",
        "allowed_roots": [str(WORKSPACE)],
        "network": "none",
        "mcp_access": "none",
        "allowed_env": [],
    },
    "budget": {
        "max_wall_time_seconds": 240,
        "max_steps": 12,
        "max_output_tokens": 8_000,
    },
    "verification": {"command": VERIFICATION, "timeout_seconds": 30},
}

codex_agent = (
    MemAgentBuilder()
    .with_name("Codex correctness analyst")
    .with_memory_provider(provider)
    .with_memory_ids(MEMORY_ID)
    .with_execution_harness("codex", meta_harness=service, config=COMMON_CONFIG)
    .build(validate=False)
)

claude_agent = (
    MemAgentBuilder()
    .with_name("Claude Code adversarial reviewer")
    .with_memory_provider(provider)
    .with_memory_ids(MEMORY_ID)
    .with_execution_harness("claude-code", meta_harness=service, config=COMMON_CONFIG)
    .build(validate=False)
)

pprint(
    [
        {"agent": codex_agent.name, "agent_id": codex_agent.agent_id, "harness": codex_agent.default_harness},
        {"agent": claude_agent.name, "agent_id": claude_agent.agent_id, "harness": claude_agent.default_harness},
    ]
)

## 4. Give the coordinator a deterministic plan

A deterministic plan is appropriate here because the workflow is known in advance and auditable. It avoids spending a third model call just to decide that two named reviewers should review. `SubTask` dictionaries are JSON-safe, so this plan can be persisted with the coordinator, including on Oracle. The final comparison is a different operation: it requires semantic synthesis, so the opt-in path attaches a native coordinator `LLMProvider`. Set `MEMORIZZ_COORDINATOR_PROVIDER` and `MEMORIZZ_COORDINATOR_MODEL` to override the inexpensive OpenAI default.

The tasks have no dependency on each other. That is intentional: blind reviews run concurrently and avoid anchoring. If a second task must inspect a first task's output, model the dependency and pass or store that evidence explicitly—do not assume dependency ordering automatically changes a prompt.

In [ ]:
DELEGATION_PLAN = [
    {
        "task_id": "codex-correctness-review",
        "description": CODEX_REVIEW_TASK,
        "assigned_agent_id": codex_agent.agent_id,
        "priority": 1,
        "dependencies": [],
    },
    {
        "task_id": "claude-adversarial-review",
        "description": CLAUDE_REVIEW_TASK,
        "assigned_agent_id": claude_agent.agent_id,
        "priority": 1,
        "dependencies": [],
    },
]

COORDINATOR_PROVIDER = os.getenv("MEMORIZZ_COORDINATOR_PROVIDER", "openai").strip().lower()
COORDINATOR_MODEL = os.getenv("MEMORIZZ_COORDINATOR_MODEL", "gpt-4.1-mini").strip()
if RUN_MULTI_HARNESS_DEMO and COORDINATOR_PROVIDER == "openai" and not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "Coordinator synthesis requires OPENAI_API_KEY, or set "
        "MEMORIZZ_COORDINATOR_PROVIDER/MEMORIZZ_COORDINATOR_MODEL for another configured provider."
    )

coordinator_builder = (
    MemAgentBuilder()
    .with_name("Multi-harness review coordinator")
    .with_instruction(
        "Coordinate independent reviews. Preserve disagreements, distinguish verified "
        "facts from recommendations, and never claim success without harness evidence."
    )
    .with_memory_provider(provider)
    .with_memory_ids(MEMORY_ID)
    .with_delegation(
        [codex_agent, claude_agent],
        enabled=True,
        mode="deterministic",
        plan=DELEGATION_PLAN,
        return_report=True,
        persist_participants=False,
    )
)
if RUN_MULTI_HARNESS_DEMO:
    coordinator_builder.with_llm_config(
        {"provider": COORDINATOR_PROVIDER, "model": COORDINATOR_MODEL}
    )
coordinator = coordinator_builder.build(validate=False)
if RUN_MULTI_HARNESS_DEMO and coordinator.model is None:
    raise RuntimeError(
        "Coordinator LLMProvider initialization failed. Check the provider extra, credentials, and model name before launching paid delegates."
    )

assert len(coordinator.delegates) == 2
assert coordinator.delegation_config["mode"] == "deterministic"
assert all(isinstance(item, dict) for item in coordinator.delegation_config["plan"])
pprint(coordinator.delegation_config["plan"])

## 5. Optionally run both harnesses

The execution cell requires **both** probes and the coordinator provider to be ready before starting either review. It calls `coordinator.run()`, not the harness service directly, so tenant scope, request context, trace identity, shared-memory workflow state, deterministic task assignment, model-based reconciliation, and partial-failure reporting all flow through the MemAgent multi-agent system.

The coordinator model is used only for reconciliation, not task planning or workspace access. If an intentionally model-less coordinator is used elsewhere, MemoRizz performs deterministic evidence-preserving aggregation and reports `consolidation.strategy="deterministic"`; it no longer logs that supported path as an error. A configured provider failure is different: the report becomes partial and records the deterministic fallback.

In [ ]:
workflow_report = None

if not RUN_MULTI_HARNESS_DEMO:
    print(
        "Safe default: the Codex + Claude Code team was composed but not executed. "
        "Set MEMORIZZ_RUN_MULTI_HARNESS_DEMO=1 before starting Jupyter to opt in."
    )
else:
    not_ready = {name: probe.get("error") for name, probe in probes.items() if not probe.get("ready")}
    if not_ready:
        raise RuntimeError(f"All panel members must be ready before execution: {not_ready}")

    workflow_report = coordinator.run(
        (
            "Perform an independent multi-harness review of the metrics ratio API. "
            "Return both reviewers' findings, agreements, disagreements, and the "
            "recommended next action."
        ),
        memory_id=MEMORY_ID,
        user_id=USER_ID,
        thread_id=THREAD_ID,
        context={
            "review_kind": "independent_blind_panel",
            "required_harnesses": required_harnesses,
            "grounding_required": True,
        },
    )
    pprint(workflow_report)

## 6. Judge the team from first-party evidence

A multi-agent coordinator's `completed` task state means a delegate returned control; it is not a substitute for vendor-run verification. The MetaHarness run ledger is the source of truth for adapter status, grounding, verification, latency, usage, cost telemetry, and errors.

The code below builds a compact panel scorecard. When the opt-in run is disabled it explains what would be checked instead of fabricating measurements.

In [ ]:
if workflow_report is None:
    print("No execution evidence exists because the opt-in run was skipped.")
else:
    run_rows = service.list_runs(limit=10)
    panel_rows = [row for row in run_rows if row.get("harness") in required_harnesses]
    scorecard = []
    for row in sorted(panel_rows, key=lambda item: item["harness"]):
        result = dict(row.get("result") or {})
        context_pack = dict(result.get("context_pack") or {})
        scorecard.append(
            {
                "harness": row["harness"],
                "run_id": row["run_id"],
                "status": row["status"],
                "verified": result.get("verified"),
                "grounded": memory_source_id in context_pack.get("source_ids", []),
                "latency_ms": result.get("latency_ms"),
                "usage": result.get("usage"),
                "cost_usd": result.get("cost_usd"),
                "error_code": result.get("error_code"),
            }
        )

    pprint(scorecard)
    pprint({"consolidation": workflow_report.get("consolidation")})
    assert workflow_report["ok"] is True
    assert workflow_report["consolidation"]["status"] == "succeeded"
    assert workflow_report["consolidation"]["model_used"] is True
    assert {row["harness"] for row in scorecard} == set(required_harnesses)
    assert all(row["status"] == "succeeded" for row in scorecard)
    assert all(row["verified"] is True for row in scorecard)
    assert all(row["grounded"] is True for row in scorecard)

## 7. Memory-first does not mean “put every answer in every prompt”

This panel uses memory at three distinct levels:

1. **Task grounding:** both harnesses receive the same bounded requirement record with a source ID.
2. **Workflow coordination:** the coordinator records scoped subtask states and participant contributions in shared memory.
3. **Continual-learning evidence:** normalized, verified runs can later support workflow canonicalization and skill promotion.

```mermaid
flowchart LR
    Requirements[(Scoped requirements)] --> C[Codex report]
    Requirements --> A[Claude report]
    C & A --> Agreement{Agreement with verification?}
    Agreement -- no --> Escalate[Human review or targeted third task]
    Agreement -- yes --> Outcome[Business-grade outcome]
    Outcome --> Repeat{Repeated diverse evidence?}
    Repeat -- no --> History[Retain as workflow history]
    Repeat -- yes --> Gates[Promotion and shadow-evaluation gates]
    Gates --> Skill[Reviewed, progressively retrieved skill]
```

Do not promote model agreement alone. Two harnesses can share the same blind spot or training prior. Host verification, downstream outcomes, repeated executions, tenant isolation, and human review remain part of the evidence contract.

## When to use this pattern

| Situation | Recommended topology | Reason |
|---|---|---|
| Routine formatting or obvious fix | One harness | A panel adds cost and little independent information |
| Security-sensitive or irreversible change | Independent implementer/reviewer roles | Correlated-error reduction can justify extra cost |
| Ambiguous root cause | Parallel specialists with different hypotheses | Preserves exploration diversity and reduces anchoring |
| One workspace write | One approved writer, then one or more read-only reviewers | Prevents concurrent edit races and keeps approval comprehensible |
| Provider outage | Route to an eligible fallback harness | Maintains service, but record that the topology changed |
| Evaluation | Fixed deterministic panel and pinned versions | Makes comparisons reproducible |

For a write workflow, do **not** change both agents above to direct mode. Use a serial plan: one exact approved writer, host verification, then a read-only reviewer. MetaHarness prevents concurrent write-capable runs from leasing the same workspace, but the architecture should express the intended ownership before that safety net is needed.

## Extending the team without losing control

Good next steps are small and evidence-driven:

- attach a configured `LLMProvider` only to the coordinator for structured reconciliation;
- add dependencies only where one task genuinely consumes another's output;
- add a third harness only after traces show a missing capability, not for a superficial vote;
- define an application-specific business outcome evaluator before workflow promotion;
- compare token, cost, latency, verified success, and unique-findings yield against the one-harness baseline; and
- persist participants only when stable identity across restarts is required.

The key design principle is simple: harness diversity is an execution strategy; MemoRizz memory, governance, and evidence remain the stable platform.

In [ ]:
coordinator.close(close_memory_provider=False)
codex_agent.close(close_memory_provider=False)
claude_agent.close(close_memory_provider=False)
service.close()
provider.close()
shutil.rmtree(DEMO_ROOT, ignore_errors=True)
print("Removed tutorial resources:", DEMO_ROOT)